In [ ]:
import numpy as np
%pip install torch
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import time

import torch

print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
"""   
DEFINIÇÃO DO CONSTRUTOR DA CLASSE

    Parâmetros de inicialização
"""
class MLPClassifier:
    def __init__(self, hidden_layers=[64, 32], activation='relu', learning_rate=0.01,
              epochs=100, patience=5, batch_size=32, optimizer='adam', regularization=None,
              dropout_p=0.1, lambda_12=0.001, random_state=None):
        
        """
        Inicializa o classificador MLP para problemas multiclasse.

         Parâmetros:
        - hidden layers: lista com número de neurônios em cada camada oculta
        - activation: função de ativação ('relu' ou 'tanh')
        - learning rate: taxa de aprendizado
        - epochs: número máximo de épocas
        - patience: paciência para early stopping
        - batch size: tamanho do batch para treinamento
        - optimizer: 'sgd' ou 'adam', o otimizador a ser usado
        - regularization: 'dropout', '12' ou None, tipo de regularização a ser aplicado
        - dropout p: probabilidade de dropout (para regularização dropout)
        - lambda 12: fator de regularização L2 (para penalização L2)
        - random state: semente para reprodutibilidade
        """

        self.hidden_layers = hidden_layers
        self.activation = activation.lower()
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.patience = patience
        self.batch_size = batch_size
        self.optimizer = optimizer.lower()
        self.regularization = regularization
        self.dropout_p = dropout_p
        self.lambda_12 = lambda_12
        self.random_state = random_state
        self.model = None
        self.loss_history = {'train': [], 'val': []}
        self.accuracy_history = {f'train': [], 'val': []}
        self.classes_ = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Usando dispositivo: {self.device}")
        print(f"Cuda habilitado? {torch.cuda.is_available()}") # Deve retornar True
        torch.cuda.empty_cache()

        if random_state is not None:
            torch.manual_seed(random_state)
            np.random.seed(random_state) 

    """   
    CONSTRUINDO AS CAMADAS DO MODELO

            Usando PyTorch
    """

    def _build_model(self, input_dim, output_dim):

        # Constrói a arquitetura do modelo PyTorch com regularização, se necessário.
        layers = []
        prev_dim = input_dim
        
        # Adiciona camadas ocultas
        for dim in self.hidden_layers:
            layers.append(nn.Linear(prev_dim, dim))
            if self.activation == 'relu':
                layers.append(nn.ReLU())
            elif self.activation == 'tanh':
                layers.append(nn.Tanh())

            # Aplica dropout, se a regularização for ‘dropout’
            if self.regularization == 'dropout':
                layers.append(nn.Dropout(p=self.dropout_p))

            prev_dim = dim

        # Camada de saida (sem ativação quando usando CrossEntropyloss)
        layers.append(nn.Linear(prev_dim, output_dim))
        return nn.Sequential(*layers).to(self.device) # Forcar uso da GPU com cuda()
    
    """ 
    IMPLEMENTAÇÃO DO MÉTODO FIT
    """
    
    def fit(self, X_train, y_train, X_val=None, y_val=None):

        # Treina o modelo MLP para classificação multiclasse
        start = time.time()

        # Converter dados para tensores PyTorch
        X_train_tensor = torch.FloatTensor(X_train).to(self.device)
        y_train_tensor = torch.LongTensor(y_train).to(self.device)


        if X_val is not None and y_val is not None:
            X_val_tensor = torch.FloatTensor(X_val).to(self.device)
            y_val_tensor = torch.LongTensor(y_val).to(self.device)
            validation_data = (X_val_tensor, y_val_tensor)
        else:
            validation_data = None

        # Inicializar modelo
        input_dim = X_train.shape[1]
        self.classes_ = torch.unique(y_train_tensor)
        output_dim = len(self.classes_)

        self.model = self._build_model(input_dim, output_dim) 

        #id = torch.cuda.current_device()
        #print(f"GPU em uso: {torch.cuda.get_device_name(id)}") # Mostra qual GPU esta sendo usada
        
        criterion = nn.CrossEntropyLoss() if output_dim > 1 else nn.BCEWithLogitsLoss()

        # Selecionar otimizador com base na escolha do usuario
        if self.optimizer == 'sgd':
            optimizer = optim.SGD(self.model.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adam':
            # Aplicar regularização L2, se for solicitado
            if self.regularization == "12":
                optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate, weight_decay=self.lambda_12)
            else:
                optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        else:
            raise ValueError("O otimizador deve ser 'sgd' ou 'adam'.")
        
        # Early stopping
        best_loss = np.inf
        patience_counter = 0
        best_weights = None

        # Treinamento
        for epoch in range(self.epochs):
            self.model.train().to(self.device) # Forçar o uso da GPU com cuda()
            epoch_loss = 0.0
            correct = 0
            total = 0

            # Mini-batch training
            for i in range(0, len(X_train), self.batch_size):
                batch_X = X_train_tensor[i:i+self.batch_size]
                batch_y = y_train_tensor[i:i+self.batch_size]

                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == batch_y).sum().item()
                total += batch_y.size(0)

            # Calcular métricas de treinamento
            train_loss = epoch_loss / (len(X_train) / self.batch_size)
            train_acc = correct / total
            self.loss_history['train'].append(train_loss)
            self.accuracy_history['train'].append(train_acc) 

            # Validação
            val_loss, val_acc = None, None
            if validation_data is not None:
                self.model.eval()
                with torch.no_grad():
                    X_val_tensor, y_val_tensor = validation_data
                    outputs = self.model(X_val_tensor)
                    val_loss = criterion(outputs, y_val_tensor).item()
                    _, predicted = torch.max(outputs.data, 1)
                    val_acc = (predicted == y_val_tensor).sum().item() / len(y_val_tensor)

                self.loss_history[ 'val'].append(val_loss)
                self.accuracy_history['val'].append(val_acc)

                # Early stopping
                if val_loss < best_loss:
                    best_loss = val_loss
                    patience_counter = 0
                    best_weights = self.model.state_dict()
                else:
                    patience_counter += 1
                    if patience_counter >= self.patience:
                        print(f"Early stopping na época {epoch+1}")
                        self.model.load_state_dict(best_weights)
                        break

            # Log de progresso
            if (epoch+1) % 10 == 0 or epoch == 0:
               msg = f"Época {epoch+1}/{self.epochs} - Treinamento Loss: {train_loss:.4f}, Treinamento Acuracia: {train_acc*100:.2f}%"
               if val_loss is not None:
                    msg += f", Validação Loss: {val_loss:.4f}, Validação Acurácia: {val_acc*100:.2f}%"
               print(msg) 

        end = time.time()
        tranning_time = end - start
        print(f"Tempo total de treinamento: {tranning_time:.2f} segundos") 

    """ 
    IMPLEMENTAÇÃO DO MÉTODO PREDICT
    """

    def predict(self, X):
        if self.model is None:
            raise RuntimeError("O modelo não foi treinado ainda. Chame fit() primeiro.")
        
        self.model .eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            outputs = self.model(X_tensor)
            _, predictions = torch.max(outputs.data, 1)
        return predictions.cpu().numpy() # CPU para evitar problemas de memória na GPU e convertê-los para numpy
    
    """ 
    IMPLEMENTAÇÃO DO MÉTODO EVALUATE
    """
        
    def evaluate(self, X, y):
        predictions = self.predict(X)
        accuracy = np.mean(predictions == y)
        return accuracy 
    
    """ 
    IMPLEMENTAÇÃO DO MÉTODO PARA PLOTAR O GRÁFICO DO TREINAMENTO

        Mostra a loss e acurácia ao longo do treinamento
    """

    def plot_training_history(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Plot da loss
        ax1.plot(self.loss_history['train'], label='Loss do Treinamento')
        if self.loss_history['val']:
            ax1.plot(self.loss_history['val'], label='Loss da Validação')
        ax1.set_title('Loss do Treinamento e Validação')
        ax1.set_xlabel('Época')
        ax1.set_ylabel('Loss')
        ax1.legend()

        # Plot da acurácia.
        ax2.plot(self.accuracy_history['train'], label='Acurácia de Treinamento')
        if self.accuracy_history['val']:
            ax2.plot(self.accuracy_history['val'], label='Acuárcia de Validação')
        ax2.set_title('Acurácia do Treinamento e Validação')
        ax2.set_xlabel('Época')
        ax2.set_ylabel('Acurácia')
        ax2.legend()

        plt.tight_layout()
        plt.show() 

    """ 
    IMPLEMENTAÇÃO DO MÉTODO PARA PLOTAR A FRONTEIRA DE DECISÃO

    Mostra a fronteira de decisão entre as diferentes features
    """
    def plot_decision_boundary(self, X, y, step=0.02):
        if X.shape[1] != 2:
            print("A fonteira de decisão só pode ser plotada para dados 2D.")
            return
        
        # Criar grid
        x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
        y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Prever para o grid
        predictions = self.predict(grid_points).reshape(xx.shape)

        # Plotar
        plt.contourf(xx, yy, predictions, alpha=0.75)
        plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o')
        plt.title('Fronteira de Decisão')
        plt.show()

In [ ]:
#dataset de Treinamento
data = np.genfromtxt('dataset_fraude.csv', delimiter=',', skip_header=1)
X= data[:, :-1] # Features
y = data[:, -1] # Labels
y = (y + 1) // 2

# Divisão treino/teste (80% treino, 20% teste)
n = int(0.8 * len(X))

X_train = X[:n]
X_test = X[n:]

y_train = y[:n]
y_test = y[n:]


# Criar e treinar o modelo
mlp = MLPClassifier(optimizer= 'adam',
                    learning_rate=0.001,
                    hidden_layers=[50, 128, 64, 4],
                    activation= 'relu',
                    regularization= 'dropout',
                    dropout_p=0.1)

# Treinar o modelo
mlp.fit(X_train, y_train)

# Avaliar no conjunto de teste
test_acc = mlp.evaluate(X_test, y_test)
print(f"\nAcurácia do Teste: {test_acc*100:.2f}%")
      
# Plotar histórico de treinamento
mlp.plot_training_history()

# Plotar fronteira de decisão (apenas para dados 2D)
mlp.plot_decision_boundary(X_train, y_train)

mlp.predict(X_test)
mlp.plot_decision_boundary(X_test, y_test) 